In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib 
import os

In [4]:
# Config
RAW_DATA_PATH = "C:/CourseWork/Dissertation Classifying grip strategies using machine learning/data/02_processed/GripFormer_preprocessed_dataset.csv"
PROCESSED_DATA_PATH = "C:/CourseWork/Dissertation Classifying grip strategies using machine learning/data/03_processed_dl/"
MODEL_ASSETS_PATH = "C:/CourseWork/Dissertation Classifying grip strategies using machine learning/models/"

# Config
SEQUENCE_ID_COL = "unique_sequence_id"
LABEL_ENCODED_COL = "label_encoded"
TEST_SET_SIZE = 0.20
VALIDATION_SET_SIZE = 0.125
RANDOM_STATE = 46
print("Configuration loaded.")

Configuration loaded.


In [5]:
with open(RAW_DATA_PATH, "r") as f:
    for i, line in enumerate(f):
        if line.strip().startswith("unique_sequence_id"):
            header_row = i
            break

df = pd.read_csv(RAW_DATA_PATH, skiprows=header_row)

df.columns = [col.strip() for col in df.columns]

print("Data loaded successfully")
print(f"Shape: {df.shape}")
df.head()

Data loaded successfully
Shape: (947200, 22)


,unique_sequence_id,timestep,label,label_encoded,indexX_unified,indexY_unified,indexZ_unified,thumbX_unified,thumbY_unified,thumbZ_unified,...,wristZ_unified,FX,FY,FZ,FVel,FAcc,MVel,MAcc,MDec,signal_grasp
0,0,0,aiming_clear_black_one,6,0.065553,0.096750,0.039112,-0.005645,0.097128,-0.002443,...,-0.120695,0.14693,0.037306,0.199919,0.035475,1.532588,0.814293,5.804851,-9.585913,1.0
1,0,1,aiming_clear_black_one,6,0.065567,0.096760,0.039006,-0.005659,0.097054,-0.002650,...,-0.120796,0.14693,0.037306,0.199919,0.035475,1.532588,0.814293,5.804851,-9.585913,1.0
2,0,2,aiming_clear_black_one,6,0.065558,0.096789,0.038924,-0.005661,0.097042,-0.002827,...,-0.120874,0.14693,0.037306,0.199919,0.035475,1.532588,0.814293,5.804851,-9.585913,1.0
3,0,3,aiming_clear_black_one,6,0.065553,0.096803,0.038829,-0.005653,0.097025,-0.002997,...,-0.120936,0.14693,0.037306,0.199919,0.035475,1.532588,0.814293,5.804851,-9.585913,1.0
4,0,4,aiming_clear_black_one,6,0.065540,0.096812,0.038733,-0.005655,0.096972,-0.003205,...,-0.120975,0.14693,0.037306,0.199919,0.035475,1.532588,0.814293,5.804851,-9.585913,1.0


In [6]:
features_to_use = [
    'indexX_unified', 'indexY_unified', 'indexZ_unified',
    'thumbX_unified', 'thumbY_unified', 'thumbZ_unified',
    'wristX_unified', 'wristY_unified', 'wristZ_unified',
    'FX', 'FY', 'FZ', 'FVel', 'FAcc', 'MVel', 'MAcc', 'MDec', 'signal_grasp'
]

print(f"Using {len(features_to_use)} features")
print(features_to_use)

Using 18 features
['indexX_unified', 'indexY_unified', 'indexZ_unified', 'thumbX_unified', 'thumbY_unified', 'thumbZ_unified', 'wristX_unified', 'wristY_unified', 'wristZ_unified', 'FX', 'FY', 'FZ', 'FVel', 'FAcc', 'MVel', 'MAcc', 'MDec', 'signal_grasp']


In [7]:
unique_sequences = df[SEQUENCE_ID_COL].unique()
print(f"Total unique sequences: {len(unique_sequences)}")

# Split sequence IDs into training+validation and test sets
train_val_ids, test_ids = train_test_split(
    unique_sequences,
    test_size=TEST_SET_SIZE,
    random_state=RANDOM_STATE
)

# Split train + validation IDs into final training and validation sets
train_ids, val_ids = train_test_split(
    train_val_ids,
    test_size=VALIDATION_SET_SIZE,
    random_state=RANDOM_STATE
)
print(f"Training sequences: {len(train_ids)}")
print(f"Validation sequences: {len(val_ids)}")
print(f"Test sequences: {len(test_ids)}")

# Create the dataframe for each set
train_df = df[df[SEQUENCE_ID_COL].isin(train_ids)].copy()
val_df = df[df[SEQUENCE_ID_COL].isin(val_ids)].copy()
test_df = df[df[SEQUENCE_ID_COL].isin(test_ids)].copy()

# Sanity Check
assert len(set(train_ids) & set(val_ids)) == 0
assert len(set(train_ids) & set(test_ids)) == 0
assert len(set(val_ids) & set(test_ids)) == 0
print("\nSanity check passed: No sequence ID overlap between sets.")


Total unique sequences: 1850
Training sequences: 1295
Validation sequences: 185
Test sequences: 370

Sanity check passed: No sequence ID overlap between sets.


In [8]:
# FIt sclaar on train data then apply it to all sets
scalar = StandardScaler()

# fit and transform the training data
train_df.loc[:, features_to_use] = scalar.fit_transform(train_df[features_to_use])

# Only transform the val and test data
val_df.loc[:, features_to_use] = scalar.transform(val_df[features_to_use])
test_df.loc[:, features_to_use] = scalar.transform(test_df[features_to_use])


print("Features scaled successfully")
train_df[features_to_use].describe()

Features scaled successfully


,indexX_unified,indexY_unified,indexZ_unified,thumbX_unified,thumbY_unified,thumbZ_unified,wristX_unified,wristY_unified,wristZ_unified,FX,FY,FZ,FVel,FAcc,MVel,MAcc,MDec,signal_grasp
count,6.630400e+05,6.630400e+05,6.630400e+05,6.630400e+05,6.630400e+05,6.630400e+05,6.630400e+05,6.630400e+05,6.630400e+05,6.630400e+05,6.630400e+05,6.630400e+05,6.630400e+05,6.630400e+05,6.630400e+05,6.630400e+05,6.630400e+05,6.630400e+05
mean,-9.224710e-17,-6.207818e-16,7.199732e-16,6.301266e-17,1.457436e-16,3.020321e-16,-3.374392e-16,-2.513648e-16,1.697484e-17,8.230225e-17,5.575977e-16,4.386024e-16,-1.255109e-16,-4.739238e-16,-9.554777e-16,-1.314950e-15,-6.217249e-16,7.750128e-17
std,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00
min,-5.183276e+00,-1.945333e+00,-1.316763e+00,-4.349728e+00,-1.866627e+00,-1.345446e+00,-6.422406e+00,-2.745854e+00,-1.721226e+00,-1.779458e+00,-1.987735e+00,-2.240133e+00,-1.433522e+00,-7.928413e+00,-2.766271e+00,-2.054761e+00,-4.386504e+00,-4.298635e+00
25%,-6.735568e-01,-7.939632e-01,-6.957142e-01,-5.795893e-01,-8.738003e-01,-6.765698e-01,-5.884937e-01,-8.004199e-01,-7.268779e-01,-4.251650e-01,-8.139492e-01,-5.480084e-01,-3.892825e-01,-4.683796e-01,-5.587828e-01,-6.891334e-01,-5.211957e-01,2.326320e-01
50%,-2.142506e-01,2.228282e-01,-5.957908e-01,-1.502829e-01,3.221408e-01,-6.160262e-01,-1.243573e-01,-8.199034e-02,-6.172171e-01,-4.673532e-02,-5.622613e-01,1.411394e-01,-2.225364e-02,6.448573e-02,-1.807838e-02,-1.093124e-01,-9.009357e-03,2.326320e-01
75%,6.003584e-01,7.382334e-01,7.601297e-01,3.570119e-01,7.612723e-01,7.185850e-01,4.931808e-01,8.221223e-01,6.794050e-01,4.142836e-01,1.041429e+00,5.980502e-01,3.633002e-01,5.828520e-01,6.617377e-01,6.837881e-01,5.320399e-01,2.326320e-01
max,6.788138e+00,4.338900e+00,2.470069e+00,9.023966e+00,4.320102e+00,2.609599e+00,6.319770e+00,3.853415e+00,2.563978e+00,5.053528e+00,1.407776e+00,1.520842e+00,1.527243e+01,2.195128e+00,2.624825e+00,3.348969e+00,2.577367e+00,2.326320e-01


In [9]:
# Reshape data for seq models

def reshape_data(df: pd.DataFrame):
    """Reshapes a dataframe into sequence based NumPy arrays"""
    X = np.stack(df.groupby(SEQUENCE_ID_COL)[features_to_use].apply(np.array))
    y = np.stack(df.groupby(SEQUENCE_ID_COL)[LABEL_ENCODED_COL].first())
    return X, y

X_train, y_train = reshape_data(train_df)
X_val, y_val = reshape_data(val_df)
X_test, y_test = reshape_data(test_df)

# print shapes to verify
print("--- Data Shapes safter Reshaping ---")
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

--- Data Shapes safter Reshaping ---
X_train shape: (1295, 512, 18)
y_train shape: (1295,)
X_val shape: (185, 512, 18)
y_val shape: (185,)
X_test shape: (370, 512, 18)
y_test shape: (370,)


In [10]:
# Save reshaped numpy arrays and the fitted scalar object for later use

np.save(os.path.join(PROCESSED_DATA_PATH, "X_train.npy"), X_train)
np.save(os.path.join(PROCESSED_DATA_PATH, 'y_train.npy'), y_train)
np.save(os.path.join(PROCESSED_DATA_PATH, 'X_val.npy'), X_val)
np.save(os.path.join(PROCESSED_DATA_PATH, 'y_val.npy'), y_val)
np.save(os.path.join(PROCESSED_DATA_PATH, 'X_test.npy'), X_test)
np.save(os.path.join(PROCESSED_DATA_PATH, 'y_test.npy'), y_test)

print(f"Processed data saved to '{PROCESSED_DATA_PATH}'")

# Save the scalar
scalar_path = os.path.join(MODEL_ASSETS_PATH, "scaler.joblib")
joblib.dump(scalar, scalar_path)
print(f"Scalar saved to '{scalar_path}'")

Processed data saved to 'C:/CourseWork/Dissertation Classifying grip strategies using machine learning/data/03_processed_dl/'
Scalar saved to 'C:/CourseWork/Dissertation Classifying grip strategies using machine learning/models/scaler.joblib'
